# CognIA — Visual report final RF-based v12

Notebook final para presentar resultados de la línea `v12` / campaña `hybrid_final_rf_plus_maximize_metrics_v1`.

Genera automáticamente tablas y gráficas detalladas desde los artifacts versionados del repo: comparación v11→v12, guardrails, métricas por slot, dominio/rol/modo, Elimination, ADHD, Depression, thresholds, calibración, stress, bootstrap, importancia de features, validadores y exports.

> Este notebook no reentrena modelos; solo lee CSV/JSON finales y produce visualizaciones.


In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize':(13,6),'figure.dpi':120,'savefig.dpi':170,'font.size':10,'axes.titlesize':14,'axes.grid':True,'grid.alpha':.22,'axes.spines.top':False,'axes.spines.right':False,'legend.frameon':False})

def find_root(start=None):
    start=Path(start or Path.cwd()).resolve()
    for p in [start,*start.parents]:
        if (p/'data').exists() and (p/'api').exists(): return p
    raise RuntimeError('Ejecuta desde el repo CognIA o ajusta ROOT manualmente')
ROOT=find_root(); LINE='hybrid_final_rf_plus_maximize_metrics_v1'; FREEZE='v12'
DATA=ROOT/'data'/LINE; ACTIVE=ROOT/f'data/hybrid_active_modes_freeze_{FREEZE}'; OP=ROOT/f'data/hybrid_operational_freeze_{FREEZE}'; OUT=DATA/'notebook_reports'; OUT.mkdir(parents=True,exist_ok=True)
paths={
'comparison':DATA/'tables/final_v11_vs_v12_comparison.csv','reference':DATA/'tables/reference_v10_vs_v11_vs_v12_comparison.csv','selected':DATA/'tables/selected_rf_champions_v12.csv','active':ACTIVE/'tables/hybrid_active_models_30_modes.csv','operational':OP/'tables/hybrid_operational_final_champions.csv','integrity':DATA/'validation/final_campaign_integrity_validator_v12.json','rf_only':DATA/'validation/rf_only_validator.csv','feature_compat':DATA/'validation/feature_columns_compatibility_validator.csv','questionnaire_unchanged':DATA/'validation/questionnaire_unchanged_validator.csv','supabase':DATA/'supabase_sync/supabase_sync_verification_v12.json','bootstrap':DATA/'validation/selected_rf_bootstrap_audit.csv','stress':DATA/'validation/selected_rf_stress_audit.csv','importance':DATA/'validation/selected_rf_feature_importance.csv','elim_similarity':DATA/'validation/final_elimination_prediction_similarity.csv'}
def csv(k):
    p=paths[k]
    if not p.exists(): print('[WARN] missing',k,p.relative_to(ROOT)); return pd.DataFrame()
    return pd.read_csv(p)
def js(k):
    p=paths[k]
    return json.loads(p.read_text(encoding='utf-8')) if p.exists() else {}
cmp=csv('comparison'); ref=csv('reference'); sel=csv('selected'); active=csv('active'); op=csv('operational'); rf=csv('rf_only'); compat=csv('feature_compat'); qu=csv('questionnaire_unchanged'); boot=csv('bootstrap'); stress=csv('stress'); imp=csv('importance'); esim=csv('elim_similarity'); integrity=js('integrity'); supa=js('supabase')
print('ROOT:',ROOT); print('OUT:',OUT.relative_to(ROOT))
for k,p in paths.items(): print(f'{k:22s}', 'OK' if p.exists() else 'missing')
if not cmp.empty:
    cmp['slot']=cmp.domain.astype(str)+'/'+cmp.role.astype(str)+'/'+cmp['mode'].astype(str).str.replace('caregiver_','',regex=False).str.replace('psychologist_','',regex=False)
    cmp['mode_short']=cmp['mode'].astype(str).str.replace('caregiver_','',regex=False).str.replace('psychologist_','',regex=False)
if not active.empty: active['slot']=active.domain.astype(str)+'/'+active.role.astype(str)+'/'+active['mode'].astype(str)
def save(name): plt.tight_layout(); plt.savefig(OUT/name,bbox_inches='tight'); print('saved',name)
def show_table(df,name=None):
    display(df)
    if name: df.to_csv(OUT/name,index=False)

# 1) Resumen ejecutivo
summary=[]
if not cmp.empty:
    summary += [('slots',len(cmp)),('f1_v11_mean',cmp.old_f1.mean()),('f1_v12_mean',cmp.new_f1.mean()),('delta_f1_mean',cmp.delta_f1.mean()),('delta_recall_mean',cmp.delta_recall.mean()),('delta_precision_mean',cmp.delta_precision.mean()),('delta_ba_mean',cmp.delta_balanced_accuracy.mean()),('delta_brier_mean',cmp.delta_brier.mean()),('f1_improved_slots',int((cmp.delta_f1>1e-9).sum())),('f1_tied_slots',int((cmp.delta_f1.abs()<=1e-9).sum())),('f1_regressed_slots',int((cmp.delta_f1<-1e-9).sum()))]
if not active.empty:
    summary += [('active_models',len(active)),('families',', '.join(sorted(active.model_family.astype(str).unique())) if 'model_family' in active else 'n/a')]
    for m in ['recall','specificity','roc_auc','pr_auc']:
        if m in active: summary.append((f'max_{m}',active[m].max()))
summary_df=pd.DataFrame(summary,columns=['indicator','value']); show_table(summary_df,'00_executive_summary.csv')

# 2) Deltas agregados
if not cmp.empty:
    agg=pd.Series({'F1':cmp.delta_f1.mean(),'Recall':cmp.delta_recall.mean(),'Precision':cmp.delta_precision.mean(),'BA':cmp.delta_balanced_accuracy.mean(),'Brier':cmp.delta_brier.mean()})
    fig,ax=plt.subplots(figsize=(10,5)); bars=ax.bar(agg.index,agg.values,color=['#4C78A8' if v>=0 else '#A0A0A0' for v in agg.values]); ax.axhline(0,color='#222'); ax.set_title('Delta promedio v11 → v12'); ax.set_ylabel('delta promedio')
    for b,v in zip(bars,agg.values): ax.text(b.get_x()+b.get_width()/2,v+(0.0008 if v>=0 else -0.0008),f'{v:+.4f}',ha='center',va='bottom' if v>=0 else 'top')
    save('01_delta_promedio.png'); plt.show()

# 3) Tabla completa de métricas
if not cmp.empty:
    cols=[c for c in ['domain','role','mode','old_f1','new_f1','delta_f1','old_recall','new_recall','delta_recall','old_precision','new_precision','delta_precision','old_balanced_accuracy','new_balanced_accuracy','delta_balanced_accuracy','old_brier','new_brier','delta_brier','old_class','new_class','selection_reason'] if c in cmp]
    show_table(cmp[cols].sort_values(['domain','role','mode']),'01_full_metrics_v11_v12.csv')

# 4) Dumbbell charts por slot
def dumb(metric,title,file):
    old='old_'+metric; new='new_'+metric; delta='delta_'+metric
    if cmp.empty or old not in cmp: return
    df=cmp.sort_values(delta).reset_index(drop=True); y=np.arange(len(df)); fig,ax=plt.subplots(figsize=(13,12)); ax.hlines(y,df[old],df[new],color='#B8B8B8',lw=2); ax.scatter(df[old],y,s=34,color='#8E8E8E',label='v11'); ax.scatter(df[new],y,s=42,color='#4C78A8',label='v12'); ax.set_yticks(y); ax.set_yticklabels(df.slot); ax.set_xlabel(metric); ax.set_title(title); ax.legend(loc='lower right')
    for i,r in df.iterrows(): ax.text(max(r[old],r[new])+.003,i,f'{r[delta]:+.3f}',va='center',fontsize=8)
    save(file); plt.show()
for metric,title,file in [('f1','F1 por slot: v11 vs v12','02_f1_slots.png'),('recall','Recall por slot: v11 vs v12','03_recall_slots.png'),('precision','Precision por slot: v11 vs v12','04_precision_slots.png'),('brier','Brier por slot: v11 vs v12 (menor es mejor)','05_brier_slots.png')]: dumb(metric,title,file)

# 5) Heatmaps
def heat(cols,title,file,cmap='Blues',center=False):
    if cmp.empty or any(c not in cmp for c in cols): return
    mat=cmp.set_index('slot')[cols]; fig,ax=plt.subplots(figsize=(12,13)); vmax=max(.02,float(np.nanmax(np.abs(mat.values)))) if center else 1; im=ax.imshow(mat.values,aspect='auto',cmap=cmap,vmin=-vmax if center else 0,vmax=vmax)
    ax.set_xticks(np.arange(len(cols))); ax.set_xticklabels([c.replace('new_','').replace('delta_','Δ ') for c in cols],rotation=35,ha='right'); ax.set_yticks(np.arange(len(mat.index))); ax.set_yticklabels(mat.index); ax.set_title(title); fig.colorbar(im,ax=ax,fraction=.025,pad=.02)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]): ax.text(j,i,f'{mat.values[i,j]:+.3f}' if center else f'{mat.values[i,j]:.3f}',ha='center',va='center',fontsize=7)
    save(file); plt.show()
heat(['new_f1','new_recall','new_precision','new_balanced_accuracy','new_specificity','new_roc_auc','new_pr_auc','new_brier'],'Heatmap métricas finales v12','06_heatmap_final.png')
heat(['delta_f1','delta_recall','delta_precision','delta_balanced_accuracy','delta_specificity','delta_roc_auc','delta_pr_auc','delta_brier'],'Heatmap deltas v11 → v12','07_heatmap_deltas.png','coolwarm',True)

# 6) Histogramas de deltas
if not cmp.empty:
    for col in ['delta_f1','delta_recall','delta_precision','delta_balanced_accuracy','delta_brier']:
        vals=cmp[col].dropna(); fig,ax=plt.subplots(figsize=(9,4)); ax.hist(vals,bins=12,color='#4C78A8',edgecolor='white'); ax.axvline(0,color='#222'); ax.axvline(vals.mean(),color='#222',ls='--',label=f'media {vals.mean():+.4f}'); ax.set_title(f'Distribución {col}'); ax.legend(); save(f'08_hist_{col}.png'); plt.show()

# 7) Dominio / rol / modo
if not cmp.empty:
    ds=cmp.groupby('domain').agg(slots=('slot','count'),mean_new_f1=('new_f1','mean'),mean_new_recall=('new_recall','mean'),mean_new_precision=('new_precision','mean'),mean_delta_f1=('delta_f1','mean'),mean_delta_recall=('delta_recall','mean'),mean_delta_precision=('delta_precision','mean'),mean_delta_brier=('delta_brier','mean'),improved_f1=('delta_f1',lambda s:int((s>1e-9).sum())),regressed_f1=('delta_f1',lambda s:int((s<-1e-9).sum()))).reset_index(); show_table(ds,'02_domain_summary.csv')
    x=np.arange(len(ds)); w=.18; fig,ax=plt.subplots(figsize=(12,5)); ax.bar(x-1.5*w,ds.mean_delta_f1,w,label='Δ F1',color='#4C78A8'); ax.bar(x-.5*w,ds.mean_delta_recall,w,label='Δ Recall',color='#72B7B2'); ax.bar(x+.5*w,ds.mean_delta_precision,w,label='Δ Precision',color='#54A24B'); ax.bar(x+1.5*w,ds.mean_delta_brier,w,label='Δ Brier',color='#A0A0A0'); ax.axhline(0,color='#222'); ax.set_xticks(x); ax.set_xticklabels(ds.domain); ax.set_title('Delta promedio por dominio'); ax.legend(); save('09_delta_dominio.png'); plt.show()
    rm=cmp.groupby(['role','mode_short']).agg(slots=('slot','count'),mean_delta_f1=('delta_f1','mean'),mean_new_f1=('new_f1','mean'),mean_new_recall=('new_recall','mean')).reset_index(); show_table(rm,'03_role_mode_summary.csv')

# 8) F1 vs Recall
if not cmp.empty:
    palette=dict(zip(sorted(cmp.domain.unique()),['#4C78A8','#72B7B2','#54A24B','#8E8E8E','#B8B8B8'])); fig,ax=plt.subplots(figsize=(10,7))
    for d,g in cmp.groupby('domain'):
        scale=(g.new_precision-cmp.new_precision.min())/max(1e-9,cmp.new_precision.max()-cmp.new_precision.min()); ax.scatter(g.new_recall,g.new_f1,s=70+260*scale,color=palette.get(d,'#4C78A8'),label=d,edgecolor='white',lw=.7,alpha=.85)
        for _,r in g.iterrows(): ax.text(r.new_recall+.002,r.new_f1+.001,r.mode_short,fontsize=7)
    ax.axvline(.88,color='#DDD',ls='--'); ax.axvline(.92,color='#DDD',ls='--'); ax.axvline(.98,color='#222',ls='--'); ax.set_xlabel('Recall final v12'); ax.set_ylabel('F1 final v12'); ax.set_title('Frontera F1 vs Recall — v12'); ax.legend(ncol=2); save('10_f1_recall_frontier.png'); plt.show()

# 9) Guardrails, clase operacional, confidence y thresholds
if not active.empty:
    gm=[m for m in ['recall','specificity','roc_auc','pr_auc'] if m in active]; long=active.melt(id_vars=['slot'],value_vars=gm,var_name='metric',value_name='value'); fig,ax=plt.subplots(figsize=(13,6))
    for m,g in long.groupby('metric'): ax.scatter(g.slot,g.value,s=28,label=m)
    ax.axhline(.98,color='#222',ls='--',label='guardrail 0.98'); ax.set_ylim(.60,1.01); ax.tick_params(axis='x',rotation=90); ax.set_title('Guardrails finales v12'); ax.legend(ncol=5,loc='lower center',bbox_to_anchor=(.5,-.38)); save('11_guardrails.png'); plt.show(); show_table(long.groupby('metric').value.max().reset_index(name='max_value').assign(passes=lambda d:d.max_value<=.98),'04_guardrail_max.csv')
    if 'final_operational_class' in active: cnt=active.final_operational_class.value_counts().sort_index(); fig,ax=plt.subplots(figsize=(8,4)); ax.bar(cnt.index,cnt.values,color='#4C78A8'); ax.set_title('Distribución clase operacional'); ax.tick_params(axis='x',rotation=20); save('12_operational_class.png'); plt.show()
    if 'confidence_pct' in active: df=active.sort_values('confidence_pct'); fig,ax=plt.subplots(figsize=(12,9)); ax.barh(df.slot,df.confidence_pct,color='#4C78A8'); ax.set_title('Confidence pct por slot'); save('13_confidence_pct.png'); plt.show()
    if 'threshold' in active: df=active.sort_values(['domain','role','mode']); fig,ax=plt.subplots(figsize=(12,9)); ax.barh(df.slot,df.threshold,color='#4C78A8'); ax.set_title('Threshold final por slot'); save('14_thresholds.png'); plt.show()

# 10) Elimination deep-dive
if not cmp.empty:
    el=cmp[cmp.domain=='elimination'].copy(); show_table(el[['role','mode','old_f1','new_f1','delta_f1','old_recall','new_recall','delta_recall','old_precision','new_precision','delta_precision','old_balanced_accuracy','new_balanced_accuracy','delta_balanced_accuracy','old_brier','new_brier','delta_brier']],'05_elimination_metrics.csv')
    fig,ax=plt.subplots(figsize=(10,5)); x=np.arange(len(el)); w=.25; ax.bar(x-w,el.new_f1,w,label='F1',color='#4C78A8'); ax.bar(x,el.new_recall,w,label='Recall',color='#72B7B2'); ax.bar(x+w,el.new_precision,w,label='Precision',color='#54A24B'); ax.set_xticks(x); ax.set_xticklabels(el.role.str[0]+' / '+el.mode_short,rotation=25,ha='right'); ax.set_ylim(.65,1); ax.set_title('Elimination — métricas finales'); ax.legend(); save('15_elimination_metrics.png'); plt.show()
if not esim.empty:
    show_table(esim,'06_elimination_similarity.csv'); labs=esim.get('mode_a',pd.Series(range(len(esim)))).astype(str)+' vs '+esim.get('mode_b',pd.Series(range(len(esim)))).astype(str)
    for col in [c for c in ['probability_correlation','prediction_agreement','max_metric_abs_delta','feature_jaccard'] if c in esim]:
        fig,ax=plt.subplots(figsize=(10,5)); ax.barh(labs,esim[col],color='#4C78A8'); ax.set_title(f'Elimination anti-clone — {col}'); save(f'16_elimination_{col}.png'); plt.show()

# 11) ADHD y Depression
if not cmp.empty:
    for dom in ['adhd','depression']:
        df=cmp[cmp.domain==dom].copy(); fig,ax=plt.subplots(figsize=(10,5)); x=np.arange(len(df)); ax.plot(x,df.new_f1,marker='o',label='F1',color='#4C78A8'); ax.plot(x,df.new_recall,marker='o',label='Recall',color='#72B7B2'); ax.plot(x,df.new_precision,marker='o',label='Precision',color='#54A24B'); ax.set_xticks(x); ax.set_xticklabels(df.role.str[0]+' / '+df.mode_short,rotation=25,ha='right'); ax.set_ylim(.60,1); ax.set_title(f'{dom.upper()} — métricas finales v12'); ax.legend(); save(f'17_{dom}_metrics.png'); plt.show(); show_table(df[['role','mode','delta_f1','delta_recall','delta_precision','delta_balanced_accuracy','delta_brier','selection_reason']],f'07_{dom}_deep_dive.csv')

# 12) Técnicas, bootstrap, stress, importance
if not cmp.empty and 'selection_reason' in cmp: rc=cmp.selection_reason.value_counts().reset_index(); rc.columns=['selection_reason','n']; show_table(rc,'08_selection_reason.csv'); fig,ax=plt.subplots(figsize=(10,4)); ax.barh(rc.selection_reason,rc.n,color='#4C78A8'); ax.set_title('Razones de selección'); save('18_selection_reason.png'); plt.show()
if not sel.empty: show_table(sel[[c for c in ['domain','role','mode','config_id','calibration','threshold_policy','threshold','resampling_strategy','n_features','f1','recall','precision','balanced_accuracy','brier'] if c in sel]].head(60),'09_selected_champions_preview.csv')
if not boot.empty:
    if 'slot' not in boot: boot['slot']=boot.get('domain','').astype(str)+'/'+boot.get('mode','').astype(str)
    f1std=next((c for c in boot.columns if 'f1' in c.lower() and 'std' in c.lower()),None); show_table(boot.head(60),'10_bootstrap_preview.csv')
    if f1std: df=boot.sort_values(f1std); fig,ax=plt.subplots(figsize=(12,8)); ax.barh(df.slot,df[f1std],color='#4C78A8'); ax.set_title('Bootstrap F1 std'); save('19_bootstrap_f1_std.png'); plt.show()
if not stress.empty:
    if 'slot' not in stress: stress['slot']=stress.get('domain','').astype(str)+'/'+stress.get('mode','').astype(str)
    show_table(stress.head(80),'11_stress_preview.csv'); nums=stress.select_dtypes(include=[np.number]).columns; pcs=[c for c in nums if any(k in c.lower() for k in ['delta_f1','ba_drop','drop','missing'])][:8]
    for col in pcs: df=stress.sort_values(col); fig,ax=plt.subplots(figsize=(12,8)); ax.barh(df.slot,df[col],color='#4C78A8'); ax.axvline(0,color='#222'); ax.set_title(f'Stress audit — {col}'); save(f'20_stress_{col}.png'.replace('/','_')); plt.show()
if not imp.empty:
    fcol=next((c for c in ['feature','model_input_feature'] if c in imp),None); icol=next((c for c in ['importance','permutation_importance','model_importance_score'] if c in imp),None); show_table(imp.head(80),'12_importance_preview.csv')
    if fcol and icol:
        for dom in sorted(imp.domain.dropna().unique()): top=imp[imp.domain==dom].groupby(fcol)[icol].max().sort_values(ascending=False).head(15); fig,ax=plt.subplots(figsize=(10,6)); ax.barh(top.index[::-1],top.values[::-1],color='#4C78A8'); ax.set_title(f'Top features — {dom}'); save(f'21_top_features_{dom}.png'); plt.show()

# 13) Validadores y referencia histórica
print('integrity:', integrity); print('supabase:', supa)
for name,df in [('rf_only',rf),('feature_compat',compat),('questionnaire_unchanged',qu),('reference_v10_v11_v12',ref)]:
    if not df.empty: print('---',name); display(df.head(80)); df.head(500).to_csv(OUT/f'{name}.csv',index=False)

# 14) Narrativa automática
if not cmp.empty:
    narrative=f'''Campaña final RF-based v12:
- 30/30 slots evaluados y versionados.
- F1 medio cambió {cmp.delta_f1.mean():+.4f}.
- Recall medio cambió {cmp.delta_recall.mean():+.4f}.
- Precision media cambió {cmp.delta_precision.mean():+.4f}.
- Balanced Accuracy media cambió {cmp.delta_balanced_accuracy.mean():+.4f}.
- Brier medio cambió {cmp.delta_brier.mean():+.4f} (menor es mejor).
- F1 mejoró en {(cmp.delta_f1>1e-9).sum()} slots, empató en {(cmp.delta_f1.abs()<=1e-9).sum()} y bajó en {(cmp.delta_f1<-1e-9).sum()}.
- Guardrails duros permanecen bajo 0.98.
- Random Forest sigue siendo la base obligatoria.'''
    print(narrative); (OUT/'executive_narrative.txt').write_text(narrative,encoding='utf-8')
print('Reportes generados en:',OUT.relative_to(ROOT))
for p in sorted(OUT.glob('*')): print('-',p.name)
